In [1]:
# Import or install Sionna
try:
    import sionna.rt
except ImportError as e:
    import os
    os.system("pip install sionna-rt")
    import sionna.rt

# Other imports
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import drjit as dr
import mitsuba as mi

no_preview = False # Toggle to False to use the preview widget


%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
    
from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, ITURadioMaterial,\
    Camera, PathSolver, InteractionType, RadioMapSolver
from sionna.rt.utils import r_hat

In [2]:
import os
import mitsuba as mi
from sionna.rt import load_scene

# ==============================================================================
# 1. XML 파일 경로 보정 및 로드
# ==============================================================================
# 원본 파일 경로 (사용자 환경에 맞게 설정)
original_xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(original_xml_path)
fixed_xml_path = os.path.join(scene_dir, "kookmin_fixed_temp.xml") # 임시 수정 파일

# 1) 원본 XML 읽기
with open(original_xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로로 치환하여 "파일 찾기 실패" 방지
#    (예: "meshes/file.ply" -> "/data1/mh/.../meshes/file.ply")
abs_mesh_path = os.path.join(scene_dir, "meshes") + "/"
xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')

# 3) 수정된 내용을 임시 파일로 저장
with open(fixed_xml_path, 'w', encoding='utf-8') as f:
    f.write(xml_content_fixed)

print(f"[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: {fixed_xml_path}")

# 4) 장면 로드
try:
    scene = load_scene(fixed_xml_path)
    print("[성공] 장면(Scene)을 성공적으로 불러왔습니다.")
except Exception as e:
    print(f"[오류] 장면 로드 실패: {e}")

# ==============================================================================
# 2. 재질(Material) 정보 확인
# ==============================================================================
if 'scene' in globals():
    print("\n" + "="*60)
    print(f"{'Object Name':<30} | {'Assigned Material':<20}")
    print("="*60)
    
    # 씬에 있는 모든 객체를 순회하며 할당된 재질 확인
    for name, obj in scene.objects.items():
        # 재질 객체가 있으면 이름 출력, 없으면 None
        mat_name = obj.radio_material.name if obj.radio_material else "None"
        print(f"{name:<30} | {mat_name:<20}")
        
    print("="*60)
    
    # 정의된 모든 재질(Radio Material) 목록 확인
    print("\n[정의된 재질 목록]")
    for mat_name, mat in scene.radio_materials.items():
        print(f" - 이름: {mat_name:<15} (Type: {mat.itu_type}, Thickness: {mat.thickness})")

[준비] 경로가 수정된 임시 XML 파일을 생성했습니다: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_temp.xml
[성공] 장면(Scene)을 성공적으로 불러왔습니다.

Object Name                    | Assigned Material   
elm__23                        | wall                
elm__24                        | roof                
elm__25                        | 8b4513              
elm__26                        | 2f4f4f              
elm__27                        | red                 
elm__28                        | white               
elm__29                        | gray                
elm__30                        | black               
elm__31                        | darkgrey            
elm__32                        | grey                
elm__33                        | lightgrey           
elm__34                        | silver              
elm__35                        | brown               
elm__36                        | d2aa6d              
elm__37                        | a58e9a              
elm_

In [3]:
# ==============================================================================
# 1. XML 경로 수정 및 안전한 로드
# ==============================================================================
# 원본 파일 및 폴더 경로 (사용자 환경)
xml_path = "/data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_itu.xml"
scene_dir = os.path.dirname(xml_path)
meshes_dir = os.path.join(scene_dir, "meshes")
temp_xml_path = os.path.join(scene_dir, "kookmin_fixed_absolute.xml")

# 1) XML 파일 읽기
with open(xml_path, 'r', encoding='utf-8') as f:
    xml_content = f.read()

# 2) 상대 경로("meshes/")를 절대 경로("/data1/.../meshes/")로 치환
if os.path.exists(meshes_dir):
    abs_mesh_path = meshes_dir + "/" if not meshes_dir.endswith("/") else meshes_dir
    xml_content_fixed = xml_content.replace('value="meshes/', f'value="{abs_mesh_path}')
    
    # 3) 임시 파일로 저장
    with open(temp_xml_path, 'w', encoding='utf-8') as f:
        f.write(xml_content_fixed)
    print(f"[설정] 경로가 수정된 임시 XML 생성: {temp_xml_path}")
else:
    raise FileNotFoundError(f"meshes 폴더를 찾을 수 없습니다: {meshes_dir}")

# 4) load_scene으로 로드
try:
    scene = load_scene(temp_xml_path)
    print("[성공] 장면(Scene) 로드 완료.")
except Exception as e:
    print(f"[치명적 오류] 장면 로드 실패: {e}")
    raise e

#

[설정] 경로가 수정된 임시 XML 생성: /data1/mh/sionna/src/sionna/rt/scenes/kookmin/kookmin_fixed_absolute.xml
[성공] 장면(Scene) 로드 완료.


In [4]:
# ==============================================================================
# 2. 도로 재질 변경 (시각화용)
# ==============================================================================
red_road_mat = ITURadioMaterial(name="red_road_mat",
                                itu_type="concrete",
                                thickness=0.2,
                                color=[1.0, 0.0, 0.0])
scene.add(red_road_mat)

target_road_id = "elm__00"
if target_road_id in scene.objects:
    scene.objects[target_road_id].radio_material = red_road_mat
    print(f"[설정] 도로({target_road_id})를 빨간색으로 변경했습니다.")

[설정] 도로(elm__00)를 빨간색으로 변경했습니다.


In [5]:
road_object_id = "elm__00"
road_positions = []

print(f"[탐색] Mitsuba Scene 내부에서 '{road_object_id}' 형상을 찾습니다...")

if hasattr(scene, 'mi_scene'):
    mi_scene = scene.mi_scene
    
    # 1. Mitsuba Scene의 모든 Shape를 순회하며 ID 매칭
    target_shape = None
    for s in mi_scene.shapes():
        if s.id() == road_object_id:
            target_shape = s
            break
    
    if target_shape: 
        try:           
            params = mi.traverse(target_shape)
                        
            if 'vertex_positions' in params:
                vertex_buffer = params['vertex_positions']
                
                # NumPy 변환 (1차원 배열: x, y, z, x, y, z ...)
                vertices_flat = np.array(vertex_buffer)
                
                # (N, 3) 형태로 변환 (x, y, z)
                if len(vertices_flat) > 0:
                    vertices = vertices_flat.reshape(-1, 3)
                    road_positions = vertices
                    print(f"  -> 추출된 도로 좌표 수: {len(road_positions)}개")
                else:
                    print("  [경고] 버퍼가 비어있습니다.")
            else:
                print(f"[오류] '{road_object_id}' 객체에 'vertex_positions' 속성이 없습니다.")

        except Exception as e:
            print(f"[오류] Vertex 추출 중 에러 발생: {e}")
    else:
        print(f"[실패] ID가 '{road_object_id}'인 Shape를 mi_scene에서 찾을 수 없습니다.")
else:
    print("[오류] scene 객체에서 'mi_scene' 속성을 찾을 수 없습니다.")

[탐색] Mitsuba Scene 내부에서 'elm__00' 형상을 찾습니다...
  -> 추출된 도로 좌표 수: 610개


In [6]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver

# 1. 설정 및 초기화
target_pos_522 = road_positions[412]
print("="*60 + f"\n[설정] 테스트 목표: {target_pos_522}\n" + "="*60)

# 기존 객체 제거
for name in ['Tx_1', 'Tx_2', 'Tx_3', 'Car_Marker']:
    if name in scene.transmitters: scene.remove(name)
if 'rx_car' in scene.receivers: scene.remove('rx_car')

# 2. 장치 배치
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[-125.663, 56.367, -181.453], [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at(target_pos_522)
    scene.add(tx)
    print(f" -> {tx_names[i]} 배치 완료.")

rx = Receiver(name="rx_car", position=target_pos_522)
rx.receive_antenna = scene.rx_array
scene.add(rx)
print(" -> 자동차(Rx) 배치 완료.")

# 3. 시뮬레이션
print("[연산] 경로 계산 시작...")
solver = PathSolver()
paths = solver(scene, max_depth=5, samples_per_src=1000000, diffuse_reflection=True, diffraction=True)

# 4. 결과 검증
a, tau = paths.cir()
if tf.size(a) > 0 and tf.reduce_sum(tf.abs(a)) > 0:
    print(f"\n✅ [성공] 전파 도달 (Amp Sum: {tf.reduce_sum(tf.abs(a)):.2e})")
else:
    print("\n❌ [실패] 전파 미도달 (장애물 또는 거리 문제)")

# 5. 시각화
cam = Camera(position=target_pos_522 + np.array([0, 100, 100]), look_at=target_pos_522)
print("[시각화] 3D 뷰어 실행")
scene.preview(paths=paths, show_devices=True)

2025-12-30 17:08:09.596725: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767082089.609364  138391 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767082089.613316  138391 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767082089.624049  138391 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767082089.624059  138391 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767082089.624060  138391 computation_placer.cc:177] computation placer alr

[설정] 테스트 목표: [ 4.1827621e+02  1.8871996e-14 -3.0820306e+02]
 -> Tx_1 배치 완료.
 -> Tx_2 배치 완료.
 -> Tx_3 배치 완료.
 -> 자동차(Rx) 배치 완료.
[연산] 경로 계산 시작...

❌ [실패] 전파 미도달 (장애물 또는 거리 문제)
[시각화] 3D 뷰어 실행


I0000 00:00:1767082092.303897  138391 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 19397 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:81:00.0, compute capability: 8.6
I0000 00:00:1767082092.305045  138391 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 22057 MB memory:  -> device: 1, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:c1:00.0, compute capability: 8.6


In [7]:
import numpy as np
import mitsuba as mi
from sionna.rt import Transmitter, PlanarArray, load_scene, Camera
import os
import matplotlib.pyplot as plt

# ==============================================================================
# 1. 초기 설정 및 데이터 준비
# ==============================================================================
output_dir = "car_move"
os.makedirs(output_dir, exist_ok=True)
print(f"[설정] 이미지는 '{output_dir}' 폴더에 저장됩니다.")

if 'road_positions' not in globals() or len(road_positions) == 0:
    raise ValueError("road_positions 데이터가 없습니다. 먼저 도로 좌표를 추출해주세요.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

waypoints = road_positions[path_indices].copy() 
waypoints[:, 1] += 1.5  

# 누적 거리 계산
diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)
cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]

speed_kmh = 60.0
speed_ms = speed_kmh * 1000.0 / 3600.0
total_time = total_distance / speed_ms
delta_t = 0.5

print(f"[시뮬레이션 정보]")
print(f" - 총 거리: {total_distance:.2f} m")
print(f" - 속도: {speed_kmh} km/h")
print(f" - 총 시간: {total_time:.2f} 초")
print(f" - 생성될 프레임 수: {int(total_time / delta_t) + 1} 장")

# ==============================================================================
# 2. 이동 함수
# ==============================================================================
def get_state_at_time(t):
    target_dist = speed_ms * t
    target_dist = np.clip(target_dist, 0, total_distance)
    
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))
    
    seg_start = cumulative_dists[idx]
    seg_len = segment_dists[idx]
    
    if seg_len == 0:
        return waypoints[idx], np.zeros(3)
    
    ratio = (target_dist - seg_start) / seg_len
    pos = waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])
    
    direction = (waypoints[idx+1] - waypoints[idx]) / seg_len
    vel = direction * speed_ms
    
    return pos, vel

# ==============================================================================
# 3. 씬 객체 배치
# ==============================================================================
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()):
        scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()):
        scene.remove(name)

start_pos, start_vel = get_state_at_time(0.0)
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

tx_car = Transmitter(name="Moving_Car", 
                     position=start_pos.tolist(),
                     orientation=[0,0,0], 
                     color=[0.0, 0.3, 1.0]) 
tx_car.transmit_antenna = scene.tx_array
scene.add(tx_car)

# 카메라 설정
cam_pos = [0, 200, -1000]
cam = Camera(position=cam_pos, look_at=[0, 0, 0])

# ==============================================================================
# 4. 시뮬레이션 루프 (오류 해결형)
# ==============================================================================
current_time = 0.0
frame_idx = 0

print("\n[렌더링 시작] 0.5초 단위로 이미지를 저장합니다...")

# 대화형 모드 끄기 (메모리 절약)
plt.ioff()

while current_time <= total_time + delta_t: 
    # 1. 위치 업데이트
    pos, vel = get_state_at_time(current_time)
    tx_car.position = pos 
    tx_car.velocity = vel
    
    # 2. 렌더링 및 저장
    try:
        # Sionna 렌더링 결과 받기
        render_result = scene.render(camera=cam, num_samples=64, resolution=[640, 480])
        
        filename = os.path.join(output_dir, f"frame_{frame_idx:04d}.png")

        # [핵심 수정] 반환된 결과가 'Figure'인지 '데이터'인지 확인하여 처리
        if hasattr(render_result, 'savefig'): 
            # Case A: 결과가 이미 그림(Figure)인 경우 (현재 오류의 원인 해결)
            render_result.savefig(filename, bbox_inches='tight', pad_inches=0.1)
            plt.close(render_result) # 반드시 닫아줘야 메모리 누수 방지
            
        else:
            # Case B: 결과가 데이터(Tensor)인 경우 (표준 방식)
            img_numpy = np.array(render_result)
            img_numpy = np.clip(img_numpy, 0.0, 1.0) # 밝기 클리핑
            
            fig = plt.figure(figsize=(8, 6))
            plt.imshow(img_numpy)
            plt.axis('off') 
            plt.title(f"Time: {current_time:.1f}s | Speed: {speed_kmh}km/h")
            plt.savefig(filename, bbox_inches='tight', pad_inches=0.1)
            plt.close(fig)

        if frame_idx % 10 == 0:
            print(f" -> Saved {filename} (Location: {pos[0]:.1f}, {pos[1]:.1f}, {pos[2]:.1f})")
            
    except Exception as e:
        print(f" [Error] Frame {frame_idx} 실패: {e}")
        plt.close('all') # 에러 발생 시 열린 창 모두 닫기
    
    current_time += delta_t
    frame_idx += 1

print("\n[완료] 모든 이미지가 'car_move' 폴더에 저장되었습니다.")

[설정] 이미지는 'car_move' 폴더에 저장됩니다.
[시뮬레이션 정보]
 - 총 거리: 1200.79 m
 - 속도: 60.0 km/h
 - 총 시간: 72.05 초
 - 생성될 프레임 수: 145 장

[렌더링 시작] 0.5초 단위로 이미지를 저장합니다...
 -> Saved car_move/frame_0000.png (Location: 418.3, 1.5, -308.2)
 -> Saved car_move/frame_0010.png (Location: 351.1, 1.5, -260.3)
 -> Saved car_move/frame_0020.png (Location: 269.0, 1.5, -247.9)
 -> Saved car_move/frame_0030.png (Location: 185.6, 1.5, -248.8)
 -> Saved car_move/frame_0040.png (Location: 103.6, 1.5, -234.4)
 -> Saved car_move/frame_0050.png (Location: 21.5, 1.5, -220.0)


/home/lab602/miniconda3/envs/sionna_env/lib/python3.11/site-packages/drjit/ast.py:838: RuntimeWarning: The AST-transforming decorator @drjit.syntax was called more than 1000 times by your program. Since transforming and recompiling Python code is a relatively expensive operation, it should not be used within loops or subroutines. Please move the function to be transformed to the top program level and decorate it there.
  warnings.warn(


 -> Saved car_move/frame_0060.png (Location: -60.3, 1.5, -204.5)
 -> Saved car_move/frame_0070.png (Location: -142.0, 1.5, -187.8)
 -> Saved car_move/frame_0080.png (Location: -93.4, 1.5, -198.0)
 -> Saved car_move/frame_0090.png (Location: -15.8, 1.5, -214.3)
 -> Saved car_move/frame_0100.png (Location: -97.8, 1.5, -199.6)
 -> Saved car_move/frame_0110.png (Location: -94.2, 1.5, -202.1)
 -> Saved car_move/frame_0120.png (Location: -28.6, 1.5, -220.5)
 -> Saved car_move/frame_0130.png (Location: -108.9, 1.5, -198.2)
 -> Saved car_move/frame_0140.png (Location: -188.7, 1.5, -174.6)

[완료] 모든 이미지가 'car_move' 폴더에 저장되었습니다.


In [8]:
import numpy as np
import tensorflow as tf
import ipywidgets as widgets
from IPython.display import display, clear_output
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver

# ==============================================================================
# 0. 데이터 준비 & 이동 경로 계산
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]

waypoints = road_positions[path_indices].copy()
waypoints[:, 1] += 1.5 

# 거리 및 시간 계산 (60km/h)
diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)
cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]
speed_ms = 60.0 / 3.6
total_time = total_distance / speed_ms
delta_t = 0.5 

def get_pos_at_time(t):
    target_dist = np.clip(speed_ms * t, 0, total_distance)
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))
    
    seg_start = cumulative_dists[idx]
    seg_len = segment_dists[idx]
    if seg_len == 0: return waypoints[idx]
    
    ratio = (target_dist - seg_start) / seg_len
    pos = waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])
    return pos  # 이미 numpy array입니다.

time_steps = np.arange(0, total_time + delta_t, delta_t)

# ==============================================================================
# 1. 씬(Scene) 초기화
# ==============================================================================
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()): scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()): scene.remove(name)

scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="VH")

tx_positions = [[-125.663, 56.367, -181.453], [323.472, 36.869, -204.315], [0.663, 56.367, -181.453]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]

for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at([0,0,0]) 
    scene.add(tx)

start_pos = get_pos_at_time(0.0)
rx = Receiver(name="rx_car", position=start_pos)
rx.receive_antenna = scene.rx_array
scene.add(rx)

solver = PathSolver()

# ==============================================================================
# 2. 인터랙티브 위젯 (수정완료)
# ==============================================================================
output_widget = widgets.Output()

def update_simulation(frame_idx):
    t = time_steps[frame_idx]
    current_pos = get_pos_at_time(t) # 여기서 이미 numpy array로 받아옵니다.
    
    # 위치 업데이트
    rx.position = current_pos
    for name in tx_names:
        scene.transmitters[name].look_at(current_pos)
    
    # 경로 계산
    paths = solver(scene, max_depth=3, samples_per_src=100000, 
                   diffuse_reflection=True, diffraction=True)
    
    with output_widget:
        output_widget.clear_output(wait=True)
        
        # 3D 뷰어
        scene.preview(paths=paths, show_devices=True, resolution=[800, 600])
        
        # 정보 출력
        a, _ = paths.cir()
        p_val = tf.reduce_sum(tf.abs(a)**2) if tf.size(a) > 0 else 0.0
        db_val = 10 * np.log10(p_val) if p_val > 0 else -np.inf
        
        # [수정] current_pos.numpy() -> current_pos (이미 numpy 배열이라 메서드 호출 불필요)
        print(f"⏱ Time: {t:.1f}s | 📍 Pos: {current_pos} | 📶 Power: {db_val:.2f} dB")

slider = widgets.IntSlider(
    value=0, min=0, max=len(time_steps)-1, step=1,
    description='Time Step:', layout=widgets.Layout(width='600px')
)

widgets.interactive_output(update_simulation, {'frame_idx': slider})

print("▼ 슬라이더를 움직여보세요.")
display(slider, output_widget)

▼ 슬라이더를 움직여보세요.


IntSlider(value=0, description='Time Step:', layout=Layout(width='600px'), max=145)

Output()

In [9]:
import numpy as np
import tensorflow as tf
from sionna.rt import Camera, PathSolver, PlanarArray, Transmitter, Receiver
import os
from tqdm import tqdm

# ==============================================================================
# 0. 설정 및 데이터 준비
# ==============================================================================
if 'road_positions' not in globals():
    raise ValueError("❌ 'road_positions' 데이터가 없습니다.")

path_indices = [
    412, 410, 408, 406, 404, 401, 400, 72, 69, 68, 342, 340, 338, 335, 334, 
    88, 86, 84, 81, 464, 462, 459, 295, 370, 294, 368, 366, 363, 308, 306, 
    303, 302, 514, 511, 510, 488, 483, 482, 172, 269, 292, 288, 286, 284, 
    281, 604, 601, 507, 289, 506, 268, 522, 517, 516, 122, 120, 117, 116, 
    525, 582
]
waypoints = road_positions[path_indices].copy()
waypoints[:, 1] += 1.5 

diffs = waypoints[1:] - waypoints[:-1]
segment_dists = np.linalg.norm(diffs, axis=1)
cumulative_dists = np.concatenate(([0], np.cumsum(segment_dists)))
total_distance = cumulative_dists[-1]
speed_ms = 60.0 / 3.6
total_time = total_distance / speed_ms
delta_t = 0.5 

def get_pos_at_time(t):
    target_dist = np.clip(speed_ms * t, 0, total_distance)
    idx = np.searchsorted(cumulative_dists, target_dist) - 1
    idx = max(0, min(idx, len(waypoints) - 2))
    seg_start, seg_len = cumulative_dists[idx], segment_dists[idx]
    if seg_len == 0: return waypoints[idx]
    ratio = (target_dist - seg_start) / seg_len
    return waypoints[idx] + ratio * (waypoints[idx+1] - waypoints[idx])

time_steps = np.arange(0, total_time + delta_t, delta_t)

# ==============================================================================
# 1. 씬(Scene) 초기화
# ==============================================================================
if hasattr(scene, 'transmitters'):
    for name in list(scene.transmitters.keys()): scene.remove(name)
if hasattr(scene, 'receivers'):
    for name in list(scene.receivers.keys()): scene.remove(name)

scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")

tx_positions = [[-125.66, 56.36, -181.45], [323.47, 36.87, -204.31], [0.66, 56.36, -181.45]]
tx_names = ["Tx_1", "Tx_2", "Tx_3"]
for i, pos in enumerate(tx_positions):
    tx = Transmitter(name=tx_names[i], position=pos, power_dbm=43)
    tx.transmit_antenna = scene.tx_array
    tx.look_at([0,0,0]) 
    scene.add(tx)

rx = Receiver(name="rx_car", position=get_pos_at_time(0.0))
rx.receive_antenna = scene.rx_array
scene.add(rx)

solver = PathSolver()

# ==============================================================================
# 2. 데이터 수집 루프
# ==============================================================================
history_a = []      
history_tau = []    
history_pos = []    
history_time = []   

print(f"🚀 시뮬레이션 시작: 총 {len(time_steps)} 프레임 계산 중...")

for t in tqdm(time_steps):
    pos = get_pos_at_time(t)
    
    rx.position = pos
    for name in tx_names:
        scene.transmitters[name].look_at(pos)
        
    paths = solver(scene, max_depth=3, samples_per_src=100000, 
                   diffuse_reflection=True, diffraction=True)
    
    # 데이터 추출 및 Numpy 변환
    if paths.a is not None:
        try:
            # 안전한 변환
            a_val = np.array(paths.a)    
            tau_val = np.array(paths.tau)
        except:
            a_val = np.array([])
            tau_val = np.array([])

        history_a.append(a_val)
        history_tau.append(tau_val)
    else:
        history_a.append(np.array([]))
        history_tau.append(np.array([]))
        
    history_pos.append(pos)
    history_time.append(t)

# ==============================================================================
# 3. 안전한 데이터 저장 (Jagged Array 처리)
# ==============================================================================
# 리스트를 바로 np.array()로 만들면 broadcasting 에러가 나므로,
# '객체(Object) 배열'을 미리 만들어서 하나씩 담습니다.

num_frames = len(history_a)
# (1) 빈 객체 배열 생성
a_objects = np.empty(num_frames, dtype=object)
tau_objects = np.empty(num_frames, dtype=object)

# (2) 하나씩 할당 (이렇게 하면 shape이 달라도 저장됨)
for i in range(num_frames):
    a_objects[i] = history_a[i]
    tau_objects[i] = history_tau[i]

save_path = 'channel_history.npz'
np.savez(save_path, 
         a=a_objects, 
         tau=tau_objects,
         pos=np.array(history_pos),
         time=np.array(history_time))

print(f"\n💾 저장 완료: {save_path}")
print("이제 2단계 코드를 실행하세요.")

🚀 시뮬레이션 시작: 총 146 프레임 계산 중...


100%|██████████| 146/146 [01:46<00:00,  1.37it/s]


💾 저장 완료: channel_history.npz
이제 2단계 코드를 실행하세요.


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ==============================================================================
# 1. 저장된 데이터 불러오기
# ==============================================================================
try:
    data = np.load('channel_history.npz', allow_pickle=True)
    hist_a = data['a']
    hist_tau = data['tau']
    hist_pos = data['pos']
    hist_time = data['time']
    print(f"✅ 데이터 로드 성공! (총 {len(hist_time)} 프레임)")
except FileNotFoundError:
    print("❌ 'channel_history.npz' 파일이 없습니다. 1단계 코드를 먼저 실행하세요.")
    raise

tx_names = ["Tx_1", "Tx_2", "Tx_3"]
NUM_TX = len(tx_names)
colors = ['r', 'g', 'b']

# ==============================================================================
# 2. 고속 인터랙티브 뷰어
# ==============================================================================
output_widget = widgets.Output()

def view_history(frame_idx):
    t = hist_time[frame_idx]
    pos = hist_pos[frame_idx]
    
    # 저장된 객체 배열에서 꺼냄 (원래 shape 유지됨)
    a_raw = hist_a[frame_idx] 
    tau_raw = hist_tau[frame_idx]
    
    with output_widget:
        output_widget.clear_output(wait=True)
        
        #  스타일 그래프 생성
        fig, ax = plt.subplots(figsize=(10, 5))
        connected = False
        max_db = -200.0
        
        if a_raw.size > 0:
            # 안전하게 1차원으로 펴서(Flatten) 데이터 처리
            powers_flat = np.abs(a_raw.flatten())**2
            delays_flat = tau_raw.flatten()
            
            total_paths = len(delays_flat)
            paths_per_tx = total_paths // NUM_TX
            
            if total_paths > 0:
                # 데이터 길이 안전장치 (Broadcast 등으로 인해 길이가 안 맞을 경우 대비)
                limit = NUM_TX * paths_per_tx
                p_data = powers_flat[:limit].reshape(NUM_TX, paths_per_tx)
                d_data = delays_flat[:limit].reshape(NUM_TX, paths_per_tx)
                
                for i in range(NUM_TX):
                    p_vals = p_data[i]
                    d_vals = d_data[i] * 1e9 # ns
                    
                    valid_idx = p_vals > 0
                    if np.any(valid_idx):
                        connected = True
                        p_valid_db = 10 * np.log10(p_vals[valid_idx] + 1e-30)
                        d_valid_ns = d_vals[valid_idx]
                        
                        local_max = np.max(p_valid_db)
                        if local_max > max_db: max_db = local_max
                        
                        markerline, stemlines, baseline = ax.stem(
                            d_valid_ns, p_valid_db,
                            linefmt=colors[i], markerfmt=colors[i]+'o',
                            label=f"{tx_names[i]}"
                        )
                        plt.setp(stemlines, 'linewidth', 1.5, 'alpha', 0.6)
                        plt.setp(markerline, 'markersize', 4)

        status = f"Max Power: {max_db:.1f} dB" if connected else "No Signal"
        ax.set_title(f"Channel State (PDP) | T={t:.1f}s | {status}")
        ax.set_xlabel("Delay [ns]")
        ax.set_ylabel("Power [dB]")
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.legend(loc='upper right')
        
        plt.show()
        print(f"📍 Position: [{pos[0]:.1f}, {pos[1]:.1f}, {pos[2]:.1f}]")

slider = widgets.IntSlider(
    value=0, min=0, max=len(hist_time)-1, step=1,
    description='History:', layout=widgets.Layout(width='600px')
)

widgets.interactive_output(view_history, {'frame_idx': slider})

print("▼ 저장된 데이터를 재생합니다.")
display(slider, output_widget)


✅ 데이터 로드 성공! (총 146 프레임)


▼ 저장된 데이터를 재생합니다.


IntSlider(value=0, description='History:', layout=Layout(width='600px'), max=145)

Output()